In [ ]:
ml project

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import log_loss, cohen_kappa_score
from sklearn.model_selection import StratifiedKFold
import warnings
warnings.filterwarnings("ignore")

# ===================================================
# CONFIG
# ===================================================
ID_COL = "trustii_id"
AGE_COL = "Age_Baseline"
SEX_COL = "Genre"
TARGET_SEVERITY = "OUTCOME SEVERITY"
TARGET_MACE = "OUTCOME MACE"

SNP_COLS_ALL = [f"SNP{i}" for i in range(1, 289)]
PRIORITY_SNP = [f"SNP{i}" for i in range(1, 76)]

# ===================================================
# LOAD DATA
# ===================================================
print("Loading data...")
train = pd.read_csv("/home/snouzlat/Téléchargements/learning_MDM/project/data/cardihack_final-train.csv")
test = pd.read_csv("/home/snouzlat/Téléchargements/learning_MDM/project/data/cardihack_final-test.csv")
dummy = pd.read_csv("/home/snouzlat/Téléchargements/learning_MDM/project/data/cardihack_dummy_pred.csv")

print(f"Train shape: {train.shape}, Test shape: {test.shape}")
print(f"MACE distribution: {train[TARGET_MACE].value_counts().to_dict()}")
print(f"Severity distribution: {train[TARGET_SEVERITY].value_counts().to_dict()}")

# ===================================================
# IMPROVED SNP FILTERING
# ===================================================
def filter_by_maf(df, snp_cols, maf_threshold=0.01):
    """Filter SNPs by Minor Allele Frequency"""
    selected = []
    for snp in snp_cols:
        vals = df[snp].dropna()
        if len(vals) < 10:
            continue
        freq = vals.value_counts()
        if len(freq) < 2:
            continue
        maf = freq.min() / freq.sum()
        if maf >= maf_threshold:
            selected.append(snp)
    return selected

def chi2_select_snps(df, target, snp_cols, p_threshold=0.1):
    """Chi-square test for SNP-target association"""
    snp_pvals = []
    for snp in snp_cols:
        tab = pd.crosstab(df[snp].fillna(-1), df[target])
        if tab.shape[0] < 2 or tab.shape[1] < 2:
            continue
        try:
            _, p, _, _ = chi2_contingency(tab)
            snp_pvals.append((snp, p))
        except:
            pass

    # Sort by p-value (ascending)
    snp_pvals.sort(key=lambda x: x[1])
    return [snp for snp, _ in snp_pvals if _ < p_threshold]

def enforce_priority(selected, priority_snps, max_total=120):
    """Ensure priority SNPs are included"""
    priority_in_selected = [s for s in selected if s in priority_snps]
    other_selected = [s for s in selected if s not in priority_snps]

    # Keep at least 20% priority SNPs
    min_priority = max(5, int(max_total * 0.2))

    if len(priority_in_selected) < min_priority:
        # Add more priority SNPs
        additional = [s for s in priority_snps if s not in priority_in_selected]
        priority_in_selected.extend(additional[:min_priority - len(priority_in_selected)])

    combined = priority_in_selected + other_selected
    return combined[:max_total]

# ===================================================
# FEATURE ENGINEERING
# ===================================================
def engineer_features(df):
    """Create interaction features"""
    df = df.copy()

    # Age bins
    if AGE_COL in df.columns:
        df['Age_bin'] = pd.cut(df[AGE_COL], bins=[0, 50, 60, 70, 100], labels=[0, 1, 2, 3])
        df['Age_squared'] = df[AGE_COL] ** 2

    # Sex encoding
    if SEX_COL in df.columns:
        df[SEX_COL] = df[SEX_COL].map({'M': 1, 'F': 0}).fillna(0.5)

    return df

train = engineer_features(train)
test = engineer_features(test)

# ===================================================
# FEATURE SELECTION - SEVERITY
# ===================================================
print("\n=== SEVERITY FEATURE SELECTION ===")
sev_snps = filter_by_maf(train, SNP_COLS_ALL, maf_threshold=0.005)  # Plus permissif
print(f"After MAF filter: {len(sev_snps)} SNPs")

sev_snps = chi2_select_snps(train, TARGET_SEVERITY, sev_snps, p_threshold=0.15)  # Plus permissif
print(f"After chi2 filter: {len(sev_snps)} SNPs")

sev_features = [AGE_COL, SEX_COL, 'Age_bin', 'Age_squared'] + sev_snps[:100]
print(f"Total severity features: {len(sev_features)}")

# ===================================================
# FEATURE SELECTION - MACE
# ===================================================
print("\n=== MACE FEATURE SELECTION ===")
clinical_cols = [c for c in train.columns
                 if c not in SNP_COLS_ALL + [ID_COL, TARGET_MACE, TARGET_SEVERITY]
                 and c in test.columns]
clinical_cols = [c for c in clinical_cols if train[c].dtype in ['float64', 'int64']]

mace_snps = filter_by_maf(train, SNP_COLS_ALL, maf_threshold=0.005)
print(f"After MAF filter: {len(mace_snps)} SNPs")

mace_snps = chi2_select_snps(train, TARGET_MACE, mace_snps, p_threshold=0.10)
print(f"After chi2 filter: {len(mace_snps)} SNPs")

mace_snps = enforce_priority(mace_snps, PRIORITY_SNP, max_total=120)
print(f"After priority enforcement: {len(mace_snps)} SNPs")

mace_features = [AGE_COL, SEX_COL, 'Age_bin', 'Age_squared'] + clinical_cols + mace_snps
mace_features = list(dict.fromkeys(mace_features))[:200]  # Remove duplicates, keep more features
print(f"Total MACE features: {len(mace_features)}")

# ===================================================
# IMPROVED MODELS
# ===================================================
print("\n=== BUILDING MODELS ===")

# Severity Model - Binary classification
sev_model = Pipeline([
    ("imp", IterativeImputer(max_iter=20, random_state=42, imputation_order='random')),
    ("sc", RobustScaler()),
    ("clf", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.3,
        C=0.5,
        max_iter=10000,
        class_weight='balanced',
        random_state=42
    ))
])

# MACE Model - Multi-class classification
mace_model = Pipeline([
    ("imp", IterativeImputer(max_iter=20, random_state=42, imputation_order='random')),
    ("sc", RobustScaler()),
    ("clf", LogisticRegression(
        penalty="elasticnet",
        solver="saga",
        l1_ratio=0.3,
        C=0.3,
        max_iter=10000,
        multi_class="multinomial",
        class_weight='balanced',
        random_state=42
    ))
])

# ===================================================
# CROSS-VALIDATION FOR THRESHOLD TUNING
# ===================================================
print("\n=== CROSS-VALIDATION ===")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

best_mace_thresholds = [0.35, 0.45]  # Default
best_score = -np.inf

# Train models on full data
print("Training Severity model...")
sev_model.fit(train[sev_features], train[TARGET_SEVERITY])

print("Training MACE model...")
mace_model.fit(train[mace_features], train[TARGET_MACE])

# Find optimal thresholds via CV
print("\nOptimizing MACE thresholds...")
for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[TARGET_MACE])):
    X_train_fold = train.iloc[train_idx]
    X_val_fold = train.iloc[val_idx]

    # Temporary models for this fold
    temp_mace = Pipeline([
        ("imp", IterativeImputer(max_iter=20, random_state=42)),
        ("sc", RobustScaler()),
        ("clf", LogisticRegression(
            penalty="elasticnet", solver="saga", l1_ratio=0.3, C=0.3,
            max_iter=10000, multi_class="multinomial", class_weight='balanced',
            random_state=42
        ))
    ])

    temp_mace.fit(X_train_fold[mace_features], X_train_fold[TARGET_MACE])

    # Try different thresholds
    mace_proba_val = temp_mace.predict_proba(X_val_fold[mace_features])

    for t1 in [0.30, 0.35, 0.40]:
        for t2 in [0.40, 0.45, 0.50]:
            if t1 >= t2:
                continue

            mace_pred_val = np.zeros(len(mace_proba_val), dtype=int)
            mace_pred_val[mace_proba_val[:, 2] > t2] = 2
            mace_pred_val[(mace_proba_val[:, 1] > t1) & (mace_proba_val[:, 2] <= t2)] = 1

            mace_w = compute_sample_weight(
                class_weight={0: 1, 1: 3, 2: 4},
                y=X_val_fold[TARGET_MACE]
            )

            kappa = cohen_kappa_score(
                X_val_fold[TARGET_MACE],
                mace_pred_val,
                weights='quadratic',
                sample_weight=mace_w
            )

            if kappa > best_score:
                best_score = kappa
                best_mace_thresholds = [t1, t2]

print(f"Best MACE thresholds: {best_mace_thresholds} (Kappa: {best_score:.4f})")

# ===================================================
# CORRECTED SCORE CALCULATION
# ===================================================
def compute_competition_score(train_df, sev_model, mace_model, mace_thresholds):
    """Compute competition score with corrected formula"""
    sev_train_pred = sev_model.predict_proba(train_df[sev_features])[:, 1]

    mace_train_proba = mace_model.predict_proba(train_df[mace_features])
    mace_train_pred = np.zeros(len(mace_train_proba), dtype=int)
    mace_train_pred[mace_train_proba[:, 2] > mace_thresholds[1]] = 2
    mace_train_pred[(mace_train_proba[:, 1] > mace_thresholds[0]) &
                    (mace_train_proba[:, 2] <= mace_thresholds[1])] = 1

    # Sample weights
    mace_w = compute_sample_weight(
        class_weight={0: 1, 1: 3, 2: 4},
        y=train_df[TARGET_MACE]
    )
    sev_w = compute_sample_weight(
        class_weight={1: 1, 0: 1.5},
        y=train_df[TARGET_SEVERITY]
    )

    # Log loss for severity
    logloss = log_loss(train_df[TARGET_SEVERITY], sev_train_pred, sample_weight=sev_w)

    # Kappa for MACE
    kappa = cohen_kappa_score(
        train_df[TARGET_MACE],
        mace_train_pred,
        weights='quadratic',
        sample_weight=mace_w
    )

    # Rescale log loss (corrected formula)
    random_ll = 0.566
    worst_ll = log_loss(
        train_df[TARGET_SEVERITY],
        (~train_df[TARGET_SEVERITY].astype(bool)).astype(float),
        sample_weight=sev_w
    )

    if logloss <= random_ll:
        rescaled = 1 - (logloss / random_ll)
    else:
        rescaled = -(logloss - random_ll) / (worst_ll - random_ll)

    score = 0.7 * kappa + 0.3 * rescaled

    print(f"\n📊 COMPETITION SCORE BREAKDOWN:")
    print(f"   Log Loss (raw): {logloss:.4f}")
    print(f"   Log Loss (rescaled): {rescaled:.4f}")
    print(f"   Cohen's Kappa: {kappa:.4f}")
    print(f"   Final Score: {score:.4f}")

    return score

# Evaluate on train set
score = compute_competition_score(train, sev_model, mace_model, best_mace_thresholds)

# ===================================================
# PREDICT FOR SUBMISSION
# ===================================================
print("\n=== GENERATING PREDICTIONS ===")
severity_pred = sev_model.predict_proba(test[sev_features])[:, 1]

mace_proba = mace_model.predict_proba(test[mace_features])
mace_pred = np.zeros(len(mace_proba), dtype=int)
mace_pred[mace_proba[:, 2] > best_mace_thresholds[1]] = 2
mace_pred[(mace_proba[:, 1] > best_mace_thresholds[0]) &
          (mace_proba[:, 2] <= best_mace_thresholds[1])] = 1

print(f"MACE predictions distribution: {np.bincount(mace_pred)}")
print(f"Severity predictions - mean: {severity_pred.mean():.3f}, std: {severity_pred.std():.3f}")

# ===================================================
# SUBMISSION
# ===================================================
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET_MACE: mace_pred,
    TARGET_SEVERITY: severity_pred
})

submission = submission[dummy.columns]
submission.to_csv("c_submission.csv", index=False)


the score on the platform is: **0.25628995733300974**
